# Fundamentals of XAI — Stroke Dataset Activity

I installed the packages needed for the Explainable Boosting Classifier and for handling the imbalanced dataset.

In [ ]:
%pip install -q interpret imbalanced-learn

I imported the libraries for exploring the data, preparing the features, training the models, and displaying the results.

In [ ]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTENC
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show

I loaded the stroke dataset into `strokeDF` and checked the first 10 rows to see what the data looks like.

In [ ]:
# Load dataset
csv_file = Path('healthcare-dataset-stroke-data.csv')
if not csv_file.exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Place healthcare-dataset-stroke-data.csv beside this notebook.')

strokeDF = pd.read_csv(csv_file)
strokeDF.head(10)

I checked the last 10 rows of the dataset.

In [ ]:
strokeDF.tail(10)

I used `info()` to check the columns, their data types, and the number of non-missing entries.

In [ ]:
strokeDF.info()

I looked at the basic statistics, including the averages and the minimum and maximum values.

In [ ]:
strokeDF.describe()

I counted the missing values in each column before preparing the dataset for training.

In [ ]:
strokeDF.isnull().sum()

I calculated the average age and plotted the age, BMI, and glucose level distributions using histograms.

In [ ]:
# Histograms
print('Mean age:', strokeDF['age'].mean())
for col in ['age', 'bmi', 'avg_glucose_level']:
    plt.figure(figsize=(6, 3))
    plt.hist(strokeDF[col].dropna(), bins=20, edgecolor='black')
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

I checked the counts for hypertension and stroke to see how the two classes are distributed.

In [ ]:
# Binary variable distributions
for col in ['hypertension', 'stroke']:
    strokeDF[col].value_counts().sort_index().plot.bar(rot=0)
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

strokeDF.groupby('stroke').size()

I removed the ID column, then split the dataset into 80% training data and 20% testing data. I kept the stroke class proportions similar in both sets.

In [ ]:
# Remove ID and split the data
strokeDF = strokeDF.drop(columns='id')
X = strokeDF.drop(columns='stroke')
y = strokeDF['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('Training:', X_train.shape, y_train.shape)
print('Testing:', X_test.shape, y_test.shape)

I prepared the numerical and categorical columns, trained a logistic regression model, and checked its predictions using accuracy and a classification report.

In [ ]:
# Logistic regression
numerical_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender', 'ever_married', 'work_type',
                        'Residence_type', 'smoking_status']
binary_features = ['hypertension', 'heart_disease']

preprocessor = ColumnTransformer([
    ('numerical', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numerical_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('binary', 'passthrough', binary_features)
])

LR_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
LR_pipeline.fit(X_train, y_train)
LR_predictions = LR_pipeline.predict(X_test)

print('Accuracy:', accuracy_score(y_test, LR_predictions))
print(classification_report(y_test, LR_predictions, zero_division=0))

I filled in missing values before using the EBM model. The replacement values came from the training data, not the test data.

In [ ]:
# Missing values for EBM
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

for col in numerical_features:
    median = X_train[col].median()
    X_train_clean[col] = X_train[col].fillna(median)
    X_test_clean[col] = X_test[col].fillna(median)

for col in categorical_features + binary_features:
    if X_train[col].isna().any() or X_test[col].isna().any():
        mode = X_train[col].mode().iloc[0]
        X_train_clean[col] = X_train[col].fillna(mode)
        X_test_clean[col] = X_test[col].fillna(mode)

I used SMOTENC to balance the training data while keeping categorical values valid. I left the test data unchanged.

In [ ]:
# Balance the training data using SMOTENC
scaler = StandardScaler()
X_train_scaled = X_train_clean.copy()
X_train_scaled[numerical_features] = scaler.fit_transform(
    X_train_clean[numerical_features]
)

smote = SMOTENC(
    categorical_features=categorical_features + binary_features,
    random_state=42
)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
X_train_balanced[numerical_features] = scaler.inverse_transform(
    X_train_balanced[numerical_features]
)
for col in binary_features:
    X_train_balanced[col] = X_train_balanced[col].astype(int)

print('Before:', y_train.value_counts().to_dict())
print('After:', pd.Series(y_train_balanced).value_counts().to_dict())

I trained the Explainable Boosting Classifier and tested its predictions. I also displayed a confusion matrix to see where the model made mistakes.

In [ ]:
# Explainable Boosting Classifier
feature_types = [
    'continuous' if col in numerical_features else 'nominal'
    for col in X_train_clean.columns
]

EBM = ExplainableBoostingClassifier(
    feature_names=X_train_clean.columns.tolist(),
    feature_types=feature_types,
    interactions=0,
    random_state=42
)
EBM.fit(X_train_balanced, y_train_balanced)
EBM_predictions = EBM.predict(X_test_clean)

print('Accuracy:', accuracy_score(y_test, EBM_predictions))
print(classification_report(y_test, EBM_predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, EBM_predictions)
plt.show()

I opened the global EBM explanation to see which features have the strongest overall influence on its predictions.

In [ ]:
# Global explanation
EBM_global = EBM.explain_global(name='EBM Global Explanation')
show(EBM_global)

I selected one record from the test set, then compared the actual stroke label with the model's prediction and probability.

In [ ]:
# Prediction for one record
patient = X_test_clean.iloc[[0]]
actual = y_test.iloc[0]
prediction = EBM.predict(patient)[0]
probability = EBM.predict_proba(patient)[0, 1]

display(patient)
print('Actual:', actual)
print('Predicted:', prediction)
print(f'Stroke probability: {probability:.2%}')

I displayed the local EBM explanation to see how the patient's features contributed to that individual prediction.

In [ ]:
# Local explanation
EBM_local = EBM.explain_local(patient, name='EBM Local Explanation')
show(EBM_local)